# 07 - Taralı Alan İhlal Tespiti (Uçtan Uca)

Pipeline: Video → Araç Tespit (YOLOv8) → Takip (ByteTrack) → Bölge Kontrolü → İhlal Tespiti

**ONEMLI:** Runtime > Change runtime type > T4 GPU secin!

In [ ]:
# Drive + kurulum
import os
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/tez_models'
VIDEO_DIR = '/content/drive/MyDrive/İstanbul trafiği kayıt'

print(f"Modeller: {os.listdir(SAVE_DIR)}")
print(f"Videolar: {os.listdir(VIDEO_DIR)}")

!pip install ultralytics shapely -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from shapely.geometry import Point, Polygon
from collections import defaultdict
from IPython.display import display, Image as IPImage
import time

# Fine-tuned model
MODEL_PATH = os.path.join(SAVE_DIR, 'istanbul_traffic_v1', 'weights', 'best.pt')
model = YOLO(MODEL_PATH)
print(f"Model yuklendi: {MODEL_PATH}")

# Video sec
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mov', '.mp4', '.avi'))]
print(f"\nMevcut videolar:")
for i, v in enumerate(video_files):
    print(f"  [{i}] {v}")

VIDEO_IDX = 0  # Degistir: hangi videoyu kullanmak istiyorsan
VIDEO_PATH = os.path.join(VIDEO_DIR, video_files[VIDEO_IDX])
print(f"\nSecilen video: {video_files[VIDEO_IDX]}")

In [ ]:
# Videonun ilk karesini goster - tarali alani isaretlemek icin
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = cap.get(cv2.CAP_PROP_FPS)
h, w = first_frame.shape[:2]
cap.release()

print(f"Video: {w}x{h}, {total_frames} kare, {video_fps:.1f} FPS")

# Ilk kareyi goster
fig, ax = plt.subplots(1, 1, figsize=(16, 9))
ax.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
ax.set_title('Tarali alani isaretlemek icin koordinatlari not alin')
ax.grid(True, alpha=0.3)

# Koordinat bulmaya yardimci grid
ax.set_xticks(range(0, w, 100))
ax.set_yticks(range(0, h, 100))
plt.tight_layout()
plt.show()

print(f"\nResim boyutu: {w} x {h}")
print("Mouse ile ustteki resimde tarali alanin kose noktalarini belirleyin.")
print("Sonraki cell'de bu koordinatlari girin.")

In [ ]:
# ============================================
# TARALI ALAN KOORDINATLARI - BURAYA GIRIN
# ============================================
# Ustteki resimden tarali alanin kose noktalarini girin
# Format: [x, y] listesi, saat yonunde
# Ornek: Bir dikdortgen alan
ZONE_POINTS = [
    [400, 350],
    [700, 350],
    [750, 550],
    [350, 550],
]
# ============================================

zone_polygon = Polygon(ZONE_POINTS)

# Kontrol: tarali alani resim uzerinde goster
fig, ax = plt.subplots(1, 1, figsize=(16, 9))
ax.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))

# Polygon ciz
pts = np.array(ZONE_POINTS + [ZONE_POINTS[0]])  # kapat
ax.fill(pts[:, 0], pts[:, 1], alpha=0.3, color='red', label='Tarali Alan')
ax.plot(pts[:, 0], pts[:, 1], 'r-', linewidth=2)

# Kose noktalarini numarala
for i, (x, y) in enumerate(ZONE_POINTS):
    ax.plot(x, y, 'ro', markersize=8)
    ax.annotate(f'P{i} ({x},{y})', (x, y), textcoords="offset points",
                xytext=(10, 10), fontsize=9, color='yellow',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))

ax.set_title('Tarali Alan - Dogru mu kontrol edin!')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Polygon alani: {zone_polygon.area:.0f} piksel^2")
print("Yanlis ise ustteki koordinatlari degistirip tekrar calistirin.")

In [ ]:
# ============================================
# IHLAL TESPIT PIPELINE
# ============================================

# Ayarlar
MIN_FRAMES_IN_ZONE = 5     # Kac kare tarali alanda kalirsa ihlal sayilsin
COOLDOWN_FRAMES = 90        # Ayni arac icin tekrar ihlal suresi
CONF_THRESHOLD = 0.35       # Arac tespit guven esigi
TRACKER_TYPE = 'bytetrack.yaml'  # 04'te en iyi cikan tracker
MAX_FRAMES = None           # None = tum video, veya sayi gir (orn: 1000)

# State machine durumlari
OUTSIDE, ENTERING, INSIDE, VIOLATION = 'outside', 'entering', 'inside', 'violation'

# Her arac icin durum takibi
track_states = defaultdict(lambda: {
    'state': OUTSIDE,
    'frames_in_zone': 0,
    'frames_outside': 0,
    'violation_triggered': False,
    'cooldown': 0,
})

# Sonuclari kaydet
violations = []         # Tespit edilen ihlaller
frame_stats = []        # Her kare icin istatistik
violation_frames = {}   # Ihlal anindaki kareler (gorsellestirme icin)

cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0
process_frames = MAX_FRAMES or total_frames

print(f"Islem basliyor: {process_frames} kare")
print(f"Tarali alan: {len(ZONE_POINTS)} nokta")
print(f"Min kare esigi: {MIN_FRAMES_IN_ZONE}")
print(f"Tracker: {TRACKER_TYPE}")
print("-" * 50)

t_start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if MAX_FRAMES and frame_count > MAX_FRAMES:
        break

    # 1. Arac tespit + takip
    results = model.track(
        frame, conf=CONF_THRESHOLD, classes=[2, 3, 5, 7],
        tracker=TRACKER_TYPE, persist=True, verbose=False
    )

    current_tracks = set()
    frame_violations = 0

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        class_names = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

        for bbox, track_id, cls_id, conf in zip(boxes, ids, classes, confs):
            current_tracks.add(track_id)
            x1, y1, x2, y2 = bbox

            # 2. Aracin alt merkez noktasi
            bottom_center = ((x1 + x2) / 2, y2)
            point = Point(bottom_center)

            # 3. Tarali alanda mi?
            is_in_zone = zone_polygon.contains(point)

            # 4. State machine guncelle
            state = track_states[track_id]

            if state['cooldown'] > 0:
                state['cooldown'] -= 1

            if is_in_zone:
                state['frames_in_zone'] += 1
                state['frames_outside'] = 0

                if state['state'] == OUTSIDE:
                    state['state'] = ENTERING
                elif state['state'] == ENTERING:
                    state['state'] = INSIDE
                elif state['state'] == INSIDE:
                    if (state['frames_in_zone'] >= MIN_FRAMES_IN_ZONE
                            and state['cooldown'] == 0):
                        state['state'] = VIOLATION
                        state['violation_triggered'] = True
                        state['cooldown'] = COOLDOWN_FRAMES

                        # 5. Ihlal kaydet
                        violation = {
                            'frame_number': frame_count,
                            'timestamp': frame_count / video_fps,
                            'track_id': int(track_id),
                            'vehicle_class': class_names.get(cls_id, f'class_{cls_id}'),
                            'confidence': float(conf),
                            'bbox': [float(x) for x in bbox],
                            'bottom_center': [float(bottom_center[0]), float(bottom_center[1])],
                            'frames_in_zone': state['frames_in_zone'],
                        }
                        violations.append(violation)
                        frame_violations += 1

                        # Ihlal karesini kaydet (ilk 20 ihlal)
                        if len(violation_frames) < 20:
                            violation_frames[len(violations)] = {
                                'frame': frame.copy(),
                                'bbox': bbox,
                                'track_id': track_id,
                                'class': class_names.get(cls_id, '?'),
                            }

                        print(f"  IHLAL #{len(violations)}: Kare {frame_count}, "
                              f"Track #{track_id}, {class_names.get(cls_id, '?')}, "
                              f"Guven: {conf:.2f}, Alanda: {state['frames_in_zone']} kare")
            else:
                state['frames_outside'] += 1
                if state['frames_outside'] >= 3:
                    state['state'] = OUTSIDE
                    state['frames_in_zone'] = 0

    # Eski track'leri temizle
    stale = set(track_states.keys()) - current_tracks
    for tid in stale:
        del track_states[tid]

    # Kare istatistigi
    frame_stats.append({
        'frame': frame_count,
        'tracks': len(current_tracks),
        'violations': frame_violations,
        'total_violations': len(violations),
    })

    # Ilerleme
    if frame_count % 200 == 0:
        elapsed = time.time() - t_start
        fps_proc = frame_count / elapsed
        print(f"  Kare {frame_count}/{process_frames} | "
              f"FPS: {fps_proc:.1f} | Ihlal: {len(violations)}")

cap.release()
elapsed = time.time() - t_start

print(f"\n{'=' * 50}")
print(f"TAMAMLANDI")
print(f"{'=' * 50}")
print(f"Islenen kare: {frame_count}")
print(f"Sure: {elapsed:.1f} saniye ({frame_count/elapsed:.1f} FPS)")
print(f"Toplam ihlal: {len(violations)}")

In [ ]:
# ============================================
# IHLAL GORUNTULERI - Galeri
# ============================================

if violation_frames:
    n = len(violation_frames)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes_flat = np.array(axes).flatten()

    for idx, (v_id, v_data) in enumerate(violation_frames.items()):
        ax = axes_flat[idx]
        frame_rgb = cv2.cvtColor(v_data['frame'], cv2.COLOR_BGR2RGB)

        # Tarali alani ciz
        zone_pts = np.array(ZONE_POINTS, dtype=np.int32)
        overlay = frame_rgb.copy()
        cv2.fillPoly(overlay, [zone_pts], (255, 0, 0))
        frame_rgb = cv2.addWeighted(overlay, 0.2, frame_rgb, 0.8, 0)
        cv2.polylines(frame_rgb, [zone_pts], True, (255, 0, 0), 2)

        # Arac bbox ciz
        x1, y1, x2, y2 = v_data['bbox'].astype(int)
        cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (255, 0, 0), 3)
        label = f"#{v_data['track_id']} {v_data['class']}"
        cv2.putText(frame_rgb, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        ax.imshow(frame_rgb)
        v_info = violations[v_id - 1]
        ax.set_title(f"Ihlal #{v_id} | Kare {v_info['frame_number']} | "
                     f"{v_info['vehicle_class']} | {v_info['timestamp']:.1f}s",
                     fontsize=10)
        ax.axis('off')

    # Bos eksenleri gizle
    for idx in range(n, len(axes_flat)):
        axes_flat[idx].axis('off')

    plt.suptitle('Tespit Edilen Ihlaller', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'results', 'violation_gallery.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Hicbir ihlal tespit edilmedi. Tarali alan koordinatlarini kontrol edin.")

In [ ]:
# ============================================
# ISTATISTIKLER VE GRAFIKLER
# ============================================
import pandas as pd

results_dir = os.path.join(SAVE_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)

# Ihlal tablosu
if violations:
    df_violations = pd.DataFrame(violations)
    print("=== IHLAL TABLOSU ===")
    print(df_violations[['frame_number', 'timestamp', 'track_id',
                          'vehicle_class', 'confidence', 'frames_in_zone']].to_string(index=False))

    # CSV kaydet
    df_violations.to_csv(os.path.join(results_dir, 'violations.csv'), index=False)
    print(f"\nKayit: {results_dir}/violations.csv")

# Arac sinifi dagilimi
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Arac sinifi dagilimi
if violations:
    class_counts = df_violations['vehicle_class'].value_counts()
    axes[0, 0].bar(class_counts.index, class_counts.values, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
    axes[0, 0].set_title('Ihlal Eden Arac Sinifi Dagilimi')
    axes[0, 0].set_ylabel('Sayi')
else:
    axes[0, 0].text(0.5, 0.5, 'Ihlal yok', ha='center', va='center', fontsize=14)
    axes[0, 0].set_title('Ihlal Eden Arac Sinifi Dagilimi')

# 2. Zaman bazli ihlal dagilimi
df_stats = pd.DataFrame(frame_stats)
axes[0, 1].plot(df_stats['frame'], df_stats['total_violations'], color='red', linewidth=1.5)
axes[0, 1].set_title('Kumulatif Ihlal Sayisi')
axes[0, 1].set_xlabel('Kare')
axes[0, 1].set_ylabel('Toplam Ihlal')
axes[0, 1].grid(True, alpha=0.3)

# 3. Takip edilen arac sayisi
axes[1, 0].plot(df_stats['frame'], df_stats['tracks'], color='steelblue', linewidth=0.5, alpha=0.7)
# Hareketli ortalama
if len(df_stats) > 30:
    axes[1, 0].plot(df_stats['frame'], df_stats['tracks'].rolling(30).mean(),
                     color='navy', linewidth=2, label='30-kare ort.')
    axes[1, 0].legend()
axes[1, 0].set_title('Takip Edilen Arac Sayisi')
axes[1, 0].set_xlabel('Kare')
axes[1, 0].set_ylabel('Arac Sayisi')
axes[1, 0].grid(True, alpha=0.3)

# 4. Ozet metrikleri
summary_text = f"""
Toplam Islenen Kare: {frame_count}
Video Suresi: {frame_count / video_fps:.1f} saniye
Islem Hizi: {frame_count / elapsed:.1f} FPS

Toplam Ihlal: {len(violations)}
Ihlal / Dakika: {len(violations) / (frame_count / video_fps / 60):.1f}

Arac Sinifi Dagilimi:
"""
if violations:
    for cls, cnt in class_counts.items():
        summary_text += f"  {cls}: {cnt} ({cnt/len(violations)*100:.0f}%)\n"

axes[1, 1].text(0.1, 0.5, summary_text, transform=axes[1, 1].transAxes,
                fontsize=12, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1, 1].set_title('Ozet')
axes[1, 1].axis('off')

plt.suptitle('Tarali Alan Ihlal Tespit Sonuclari', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'violation_stats.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTum sonuclar kaydedildi: {results_dir}/")